In [ ]:
# I start by loading the data from the 2022_Building_Energy_ECM_and_Benchmarking_20240407.xlsx file

import pandas as pd

file_path = '2022_Building_Energy_ECM_and_Benchmarking_20240407.xlsx'
df = pd.read_excel(file_path)

In [ ]:
# I create the empty column 'Actual_Carbon_Savings_Greater_Than_Expected', and I fill it with 1 when the
# actual carbon savings are bigger than the expected ones, and 0 when they are not

df['Actual_Carbon_Savings_Greater_Than_Expected'] = pd.Series([])
df['Actual_Carbon_Savings_Greater_Than_Expected'] = (df['Actual_Annual_Carbon_Savings'] >= df['Expected_Annual_Carbon_Savings']).astype(int)
#print(df)

In [ ]:
# To clean the data, I drop rows that have NA value for what I selected as best numerical predictor variables

df.dropna(subset=['Actual_Carbon_Savings_Greater_Than_Expected','Actual_Annual_Carbon_Savings', 'Expected_Annual_Carbon_Savings', 'CouncilDistrictCode', 'YearBuilt', 'NumberofFloors', 'ENERGYSTARScore', 'SiteEUI(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'Electricity(kWh)', 'TotalGHGEmissions', 'ECM_Cost', 'EUL', 'Expected_Annual_Cost_Savings'], inplace=True)
print('rows remaining:', len(df))

In [ ]:
# Now I use train/test split to create the classifier, and I test its accuracy  

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Select features and target variable
X = df[['CouncilDistrictCode', 'YearBuilt', 'NumberofFloors', 'ENERGYSTARScore', 'SiteEUI(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'Electricity(kWh)', 'TotalGHGEmissions', 'ECM_Cost', 'EUL', 'Expected_Annual_Cost_Savings', 'Expected_Annual_Carbon_Savings']]  # Features
y = df['Actual_Carbon_Savings_Greater_Than_Expected']  # Target variable

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Choose and train a classifier
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the classifier
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
# I load the data from 2024_Building_ECM_Estimates.xlsx file

file_path2 = '2024_Building_ECM_Estimates.xlsx'
df2 = pd.read_excel(file_path2)
#print(df2)

In [ ]:
# I create a new X value that I use to predict the probability success - when actual carbon savings are greater than 
# expected - in the 2024 data. Since the 'predict_proba' command gives a 2-D array, I chose the highest probability
# from each row

X1 = df2[['CouncilDistrictCode', 'YearBuilt', 'NumberofFloors', 'ENERGYSTARScore', 'SiteEUI(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'Electricity(kWh)', 'TotalGHGEmissions', 'ECM_Cost', 'EUL', 'Expected_Annual_Cost_Savings', 'Expected_Annual_Carbon_Savings']]  
df2['New_Actual_Carbon_Savings_Greater_Than_Expected'] = classifier.predict(X1)
prob_carbon_savings_success_array = classifier.predict_proba(X1)
df2['prob_carbon_savings_success'] = prob_carbon_savings_success_array[:, 1] 
#print(len(df2['prob_carbon_savings_success']))
#print(df2['prob_carbon_savings_success'])

In [ ]:
# Lastly, I create a boolean selector to filter the projects that have a probability of success higher than 0.85.
# Even though it was asked to have 0.8 as limit, the value 0.85 was decided to satisfy the requirements for 
# total ECM cost, total carbon savings cost, and total simple payback. These three were calculated by summing
# the values from the filtered data set and then diving them by each other. To show my results, I print the ID
# of the selected projects, and their total ECM cost, total carbon savings cost, and total simple payback.

bool_sel1 = df2['prob_carbon_savings_success'] >= 0.85
filtered_df = df2[bool_sel1]
print('Project ID:', filtered_df['OSEBuildingID'])

total_ECM_cost = filtered_df['ECM_Cost'].sum()
print('total ECM cost:', total_ECM_cost)

expected_total_carbon_savings = filtered_df['Expected_Total_Carbon_Savings'].sum()
total_carbon_savings_cost = total_ECM_cost/expected_total_carbon_savings
print('total carbon savings cost:', total_carbon_savings_cost)

expected_annual_cost_savings = filtered_df['Expected_Annual_Cost_Savings'].sum()
total_simple_payback = total_ECM_cost/expected_annual_cost_savings
print('total simple payback:', total_simple_payback)